## Import libraries

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.compose import ColumnTransformer

# Models
from sklearn.linear_model import LinearRegression, LassoCV, Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor

# Metrics
from sklearn.metrics import mean_squared_error, r2_score

## Load the dataset

In [ ]:
df = pd.read_csv('forest+fires/forestfires.csv')
df.head()

# X = df.drop('y', axis=1)
# y = df['y']

## Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Preprocessing

In [ ]:
num_features = X.columns

trans = ColumnTransformer([
    ('num', StandardScaler(), num_features)
])

## Pipeline

In [ ]:

pipe = Pipeline([
    ('transform', trans),
    ('scale', MinMaxScaler()),   # optional (can remove redundancy)
    ('clf', LinearRegression())
])

# Grid parameters for different models
grid_param = [

    # Decision Tree
    {
        "clf": [DecisionTreeRegressor()],
        "clf__splitter": ["best", "random"],
        "clf__max_depth": [5, 7, 10],
        "clf__min_samples_leaf": [2, 4],
    },

    # Lasso (L1)
    {
        "clf": [LassoCV(cv=5)]
    },

    # Ridge (L2)
    {
        "clf": [Ridge()],
        "clf__alpha": [0.1, 1.0, 10.0]
    },

    # ElasticNet (L1 + L2)
    {
        "clf": [ElasticNet()],
        "clf__alpha": [0.1, 1.0],
        "clf__l1_ratio": [0.3, 0.5, 0.7]
    },

    # Support Vector Regressor
    {
        "clf": [SVR()],
        "clf__kernel": ['linear', 'rbf', 'poly'],
        "clf__C": [0.1, 1, 10]
    },

    # AdaBoost
    {
        "clf": [AdaBoostRegressor()],
        "clf__n_estimators": [100, 200, 400],
        "clf__learning_rate": [0.01, 0.1],
        "clf__random_state": [1]
    },

    # Random Forest
    {
        "clf": [RandomForestRegressor()],
        "clf__n_estimators": [100, 200],
        "clf__max_depth": [10, 20],
        "clf__min_samples_split": [2, 10],
        "clf__min_samples_leaf": [1, 4],
        "clf__max_features": ['sqrt'],
        "clf__random_state": [1]
    }
]



## Training the model using GridSearchCV

In [ ]:

# Grid SearchCV
grid = GridSearchCV(
    pipe,
    grid_param,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)


## Evaluation and Results

In [ ]:
# Results
print("Best Model:", grid.best_estimator_)
print("Best Params:", grid.best_params_)

# Evaluation
y_pred = grid.predict(X_test)

print("R2 Score:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))